<a href="https://colab.research.google.com/github/Rashi-Dwivedi1812/Forged-Signature-Verification/blob/main/fastapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone a fresh copy of the repository
%rm -rf Forged-Signature-Verification
!git clone https://github.com/Rashi-Dwivedi1812/Forged-Signature-Verification.git

# Navigate into the client directory
%cd Forged-Signature-Verification/client

# Install dependencies and build the project
!npm install
!npm run build

# Go back to the root directory
%cd /content/

print("\n✅ Frontend build complete!")


Cloning into 'Forged-Signature-Verification'...
remote: Enumerating objects: 1374, done.
remote: Counting objects: 100% (1374/1374), done.
remote: Compressing objects: 100% (1053/1053), done.
remote: Total 1374 (delta 269), reused 1333 (delta 239), pack-reused 0 (from 0)
Receiving objects: 100% (1374/1374), 12.55 MiB | 15.11 MiB/s, done.
Resolving deltas: 100% (269/269), done.
/content/Forged-Signature-Verification/client
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 208 packages, and audited 209 packages in 8s
⠸
⠸45 packages are looking for funding
⠸  run `npm fund` for details
⠸
2 vulnerabilities (1 moderate, 1 high)

To address all issues, run:
  npm audit fix

Run `npm audit` for details.
⠸
> client@0.0.0 build
> vite build

vite v7.1.4 building for production...
✓ 1674 modules transformed.
dist/index.html                   0.46 kB │ gzip:  0.30 kB
dist/assets/index-BKOCpneW.css   61.85 kB │ gzip:  8.79 kB
dist/assets/index-DKtovhfV.js   247.08 kB │ gzip: 70

In [2]:
# --- Install necessary Python libraries ---
!pip install fastapi uvicorn python-multipart pyngrok Pillow numpy tensorflow nest_asyncio

import tensorflow as tf
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.staticfiles import StaticFiles
from starlette.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import numpy as np
import io
import uvicorn
from pyngrok import ngrok
import os
import threading
import nest_asyncio
nest_asyncio.apply()

# --- Load Model ---
model_path = 'Forged-Signature-Verification/temp_model.h5'
try:
    # Load the full model for single predictions
    full_model = tf.keras.models.load_model(model_path)
    print(f"--- Full model '{model_path}' loaded successfully! ---")

    # Create a feature extractor model for comparison.
    feature_layer_output = full_model.layers[-2].output
    feature_extractor_model = tf.keras.Model(inputs=full_model.input, outputs=feature_layer_output)
    print("--- Feature extractor model created successfully! ---")

except Exception as e:
    print(f"--- 🔴 Error loading model: {e} ---")
    full_model = None
    feature_extractor_model = None

# --- Initialize App and Add CORS ---
app = FastAPI()

origins = ["*"]
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- Helper Function to Preprocess Images ---
def preprocess_image(image_bytes):
    """Takes image bytes and preprocesses it for the model."""
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img = img.resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

# --- API Endpoint for Single Prediction ---
@app.post("/api/predict-single")
async def predict_signature(file: UploadFile = File(...)):
    if not full_model:
        raise HTTPException(status_code=500, detail="Model not loaded")

    contents = await file.read()
    img_array = preprocess_image(contents)

    prediction = full_model.predict(img_array)
    score = float(prediction[0][0])

    is_forged_result = score > 0.5
    response_data = {
        "filename": file.filename,
        "confidence": score,
        "is_forged": is_forged_result
    }
    return response_data

# --- API Endpoint for Signature Comparison ---
@app.post("/api/compare-signatures")
async def compare_signatures(files: list[UploadFile] = File(...)):
    if not feature_extractor_model:
        raise HTTPException(status_code=500, detail="Feature extractor model not loaded")

    if len(files) != 2:
        raise HTTPException(status_code=400, detail="Please upload exactly two files for comparison.")

    # Preprocess both images
    image_one_bytes = await files[0].read()
    image_two_bytes = await files[1].read()

    img_array_1 = preprocess_image(image_one_bytes)
    img_array_2 = preprocess_image(image_two_bytes)

    # Extract feature vectors from both images
    features_1 = feature_extractor_model.predict(img_array_1)
    features_2 = feature_extractor_model.predict(img_array_2)

    # Calculate the similarity score (Euclidean distance)
    distance = np.linalg.norm(features_1 - features_2)
    similarity_score = max(0, 1 - (distance / 10))

    # Format the response for the frontend
    is_match_result = similarity_score > 0.5
    response_data = {
        "filenames": [files[0].filename, files[1].filename],
        "similarity_score": float(similarity_score),
        # ✅ FIX: Convert the numpy boolean to a standard Python boolean
        "is_match": bool(is_match_result)
    }
    return response_data

# --- Serve Frontend ---
build_dir = "Forged-Signature-Verification/client/dist"
app.mount("/assets", StaticFiles(directory=os.path.join(build_dir, "assets")), name="assets")
@app.get("/{full_path:path}")
async def serve_react_app(full_path: str):
    index_path = os.path.join(build_dir, "index.html")
    if os.path.exists(index_path):
        return FileResponse(index_path)
    return {"error": "index.html not found"}

# --- Run Server with ngrok ---
print("--- Starting server... ---")
ngrok.kill()
NGROK_AUTH_TOKEN = "32umhROMlSlB9ys4vK8jmf25SAP_37TzSffqyBmksNZg8JU8M" # Make sure your token is pasted here
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print(f"✅ Your app is live at: {public_url}")

# Run uvicorn in a separate thread to avoid asyncio loop conflicts
thread = threading.Thread(target=uvicorn.run, kwargs={"app": app, "host": "0.0.0.0", "port": 8000})
thread.start()



--- Full model 'Forged-Signature-Verification/temp_model.h5' loaded successfully! ---
--- Feature extractor model created successfully! ---
--- Starting server... ---
✅ Your app is live at: NgrokTunnel: "https://d54c2a919422.ngrok-free.app" -> "http://localhost:8000"
